# RepE Science Emergence — Colab GPU Pipeline
**Phases 7–12**: Activation collection → LAT training → Evaluation → Control generation → SAE bridge

**Before running**: Set `HF_TOKEN` and `OPENAI_API_KEY` in Colab Secrets (🔑 icon in left sidebar).

## 0. Runtime check + setup

In [ ]:
# Verify GPU is available
import torch
assert torch.cuda.is_available(), 'No GPU detected. Runtime > Change runtime type > T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Install dependencies pinned in the repo's requirements.txt.
# We do this AFTER the git clone in the next-but-one cell, but that's fine —
# Colab keeps state between cells. If you re-run from the top, this cell
# will harmlessly install before the clone (the file is also in this repo's
# parent and Colab's PYTHONPATH).
!pip install -q transformers accelerate huggingface_hub openai python-dotenv \
                scikit-learn safetensors matplotlib tqdm pydantic pyyaml

In [ ]:
# Load secrets from Colab secret manager
import os
from google.colab import userdata

os.environ['HF_TOKEN']       = userdata.get('HF_TOKEN')
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# Optional — only needed for the final "push results to GitHub" cell.
try:
    os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
    print('Secrets loaded (HF_TOKEN, OPENAI_API_KEY, GITHUB_TOKEN).')
except Exception:
    print('Secrets loaded (HF_TOKEN, OPENAI_API_KEY). GITHUB_TOKEN not set — '
          'the final push cell will be skipped.')

In [ ]:
# Mount Google Drive (outputs saved here — survives session crashes)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/repe_outputs'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Drive output root: {DRIVE_OUTPUT}')

In [ ]:
# Clone or update repo
REPO = 'https://github.com/TanmayAgarwal123/CRIS_Labs_Spring_2026.git'
BRANCH = 'repe-science-emergence'
SUBDIR = 'science-repe-emergence'

if not os.path.exists('/content/CRIS_Labs_Spring_2026'):
    !git clone --branch {BRANCH} {REPO} /content/CRIS_Labs_Spring_2026
else:
    !git -C /content/CRIS_Labs_Spring_2026 pull

PROJECT_DIR = f'/content/CRIS_Labs_Spring_2026/{SUBDIR}'
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

In [ ]:
# Symlink outputs/ to Drive so all output is persistent
import subprocess

local_outputs = os.path.join(PROJECT_DIR, 'outputs')
drive_outputs = os.path.join(DRIVE_OUTPUT, 'outputs')
os.makedirs(drive_outputs, exist_ok=True)

if os.path.islink(local_outputs):
    print('outputs/ symlink already exists.')
elif os.path.isdir(local_outputs):
    # Move existing outputs to Drive then symlink
    !cp -r {local_outputs}/* {drive_outputs}/ 2>/dev/null || true
    !rm -rf {local_outputs}
    !ln -s {drive_outputs} {local_outputs}
    print(f'Moved outputs/ to Drive and symlinked.')
else:
    !ln -s {drive_outputs} {local_outputs}
    print(f'Symlinked outputs/ → {drive_outputs}')

In [ ]:
# Sanity check: Phase 6 outputs exist AND Phase 7+ scripts are implemented.
import importlib, importlib.util, pkgutil

required_files = [
    'data/processed/train.jsonl',
    'data/processed/val.jsonl',
    'data/processed/test.jsonl',
    'data/processed/paraphrase_test.jsonl',
    'data/repe_suffixes/source_a_pairs.jsonl',
]
required_scripts = [
    'src.repe.collect_activations',
    'src.repe.train_lat_vectors',
    'src.repe.evaluate_reading_vectors',
    'src.repe.run_control_generation',
    'src.analysis.evaluate_generations',
    'src.analysis.sae_alignment',
    'src.analysis.plot_scaling_trends',
]

all_ok = True
print('Phase 6 input files:')
for p in required_files:
    ok = os.path.exists(p)
    print(f"  {'OK ' if ok else 'MISS'}  {p}")
    if not ok: all_ok = False

print('\nPhase 7+ scripts (must define main()):')
for m in required_scripts:
    try:
        mod = importlib.import_module(m)
        has_main = callable(getattr(mod, 'main', None))
    except Exception as e:
        has_main = False
        print(f"  STUB  {m}  ({e.__class__.__name__})")
        all_ok = False
        continue
    print(f"  {'OK  ' if has_main else 'STUB'}  {m}")
    if not has_main: all_ok = False

assert all_ok, (
    'One or more inputs are missing or scripts are still docstring stubs. '
    'Implement them in the repo first, push, and re-pull before re-running '
    'Phase 7+.'
)

---
## Phase 7 — Collect Activations
Runs the Qwen3 models forward over Source A (vgel-style suffix pairs) and Source B (GPT-generated pairs). Saves per-layer diff vectors. **This is the longest step.**

In [ ]:
# Choose which model to run this session.
# AGENTS.md spec: 0.6B first, then 1.7B, then 4B (each in a separate session).
# Re-run this cell + Phases 7+8+9 cells with the next MODEL_KEY each time.

MODEL_KEY  = '1.7B'    # one of '0.6B', '1.7B', '4B'

MODEL_MAP = {
    '0.6B': ('Qwen/Qwen3-0.6B-Base', 'qwen3_0p6b'),
    '1.7B': ('Qwen/Qwen3-1.7B-Base', 'qwen3_1p7b'),
    '4B':   ('Qwen/Qwen3-4B-Base',   'qwen3_4b'),
}
MODEL_NAME, MODEL_SHORT = MODEL_MAP[MODEL_KEY]
print(f'Model: {MODEL_NAME}  (short: {MODEL_SHORT})')

# Memory hint: 0.6B fits easily on T4 (16 GB); 1.7B fits in bf16; 4B is
# tight on T4 — prefer L4 or A100 if available.

In [ ]:
# Phase 7 — collect activations on TRAIN.
# Source A (vgel-style persona pairs) + Source B (GPT-generated pairs) are
# merged inside the script; per-layer diffs (h_pos[-1] - h_neg[-1]) are
# saved, tagged with `source` so we can ablate later.
!python -m src.repe.collect_activations \
    --model            {MODEL_NAME} \
    --source_b_dataset data/processed/train.jsonl \
    --source_a_pairs   data/repe_suffixes/source_a_pairs.jsonl \
    --output           outputs/activations/{MODEL_SHORT}/train.pt \
    --batch_size       1 \
    --dtype            bfloat16

In [ ]:
# Phase 7 — collect activations on VAL (Source B only; used for LAT sign-correction in Phase 8).
!python -m src.repe.collect_activations \
    --model            {MODEL_NAME} \
    --source_b_dataset data/processed/val.jsonl \
    --output           outputs/activations/{MODEL_SHORT}/val.pt \
    --batch_size       1 \
    --dtype            bfloat16

In [ ]:
# Phase 7 — collect activations on TEST + PARAPHRASE_TEST (Source B only).
for split in ['test', 'paraphrase_test']:
    !python -m src.repe.collect_activations \
        --model            {MODEL_NAME} \
        --source_b_dataset data/processed/{split}.jsonl \
        --output           outputs/activations/{MODEL_SHORT}/{split}.pt \
        --batch_size       1 \
        --dtype            bfloat16

In [ ]:
# Verify activation files were written
import glob
files = glob.glob(f'outputs/activations/{MODEL_SHORT}/*.pt')
for f in sorted(files):
    size_mb = os.path.getsize(f) / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')

---
## Phase 8 — Train LAT Vectors
Runs PCA on the diff vectors per layer. Fast (seconds). Produces one `.pt` vector file per (model, dataset).

In [ ]:
!python -m src.repe.train_lat_vectors \
    --activations  outputs/activations/{MODEL_SHORT}/train.pt \
    --val          outputs/activations/{MODEL_SHORT}/val.pt \
    --output       outputs/vectors/{MODEL_SHORT} \
    --pca_components 1 \
    --normalize

In [ ]:
# Check vectors written
import torch
vec_files = glob.glob(f'outputs/vectors/{MODEL_SHORT}/*.pt')
for f in sorted(vec_files):
    d = torch.load(f, map_location='cpu')
    print(f"  {os.path.basename(f)}")
    print(f"    layers: {len(d['vectors'])}  dim: {d['vectors'][0].shape}")
    print(f"    top explained var: {max(d['pca_explained_variance']):.3f}")

---
## Phase 9 — Evaluate Reading Vectors
Computes accuracy, AUC, and margin on test and paraphrase_test. Also runs cross-dataset transfer evaluation.

In [ ]:
!python -m src.repe.evaluate_reading_vectors \
    --vectors      outputs/vectors/{MODEL_SHORT} \
    --test         outputs/activations/{MODEL_SHORT}/test.pt \
    --paraphrase   outputs/activations/{MODEL_SHORT}/paraphrase_test.pt \
    --output       outputs/evaluations/{MODEL_SHORT} \
    --n_boot       1000

In [ ]:
# Print headline results
import json, glob

eval_files = glob.glob(f'outputs/evaluations/{MODEL_SHORT}/*.json')
for f in sorted(eval_files):
    d = json.load(open(f))
    print(f"\n{os.path.basename(f)}")
    print(f"  best layer:   {d.get('best_layer')}")
    print(f"  accuracy:     {d.get('accuracy_mean', 0):.3f}  [{d.get('accuracy_ci_lo', 0):.3f}–{d.get('accuracy_ci_hi', 0):.3f}]")
    print(f"  paraphrase:   {d.get('paraphrase_accuracy_mean', 0):.3f}")
    print(f"  AUC:          {d.get('auc_mean', 0):.3f}  [{d.get('auc_ci_lo', 0):.3f}–{d.get('auc_ci_hi', 0):.3f}]")

---
## Phase 10 — Control Generation
Runs inference-time steering on **one model only** (1.7B recommended). 10 prompts × 5 coefficients = 50 generations. Change `CONTROL_MODEL_KEY` if you want to use 4B instead.

In [ ]:
CONTROL_MODEL_KEY = '1.7B'   # or '4B'
_, CONTROL_MODEL_SHORT = MODEL_MAP[CONTROL_MODEL_KEY]
CONTROL_MODEL_NAME, _ = MODEL_MAP[CONTROL_MODEL_KEY]

# Only run if vectors exist for control model
assert os.path.exists(f'outputs/vectors/{CONTROL_MODEL_SHORT}'), \
    f'No vectors for {CONTROL_MODEL_SHORT} — run Phases 7+8 for that model first'

In [ ]:
!python -m src.repe.run_control_generation \
    --model        {CONTROL_MODEL_NAME} \
    --vectors      outputs/vectors/{CONTROL_MODEL_SHORT} \
    --prompts      configs/control_prompts.yaml \
    --coefficients "-2.0,-1.0,0.0,1.0,2.0" \
    --layers       middle_late \
    --max_new_tokens 160 \
    --output       outputs/generations/{CONTROL_MODEL_SHORT} \
    --dtype        bfloat16

In [ ]:
# Phase 11 — LLM-judge evaluation of generations (uses gpt-4.1-mini).
# Takes the generations directory written in Phase 10 and writes per-pair
# rubric scores plus aggregated by_coefficient averages.
!python -m src.analysis.evaluate_generations \
    --input        outputs/generations/{CONTROL_MODEL_SHORT} \
    --output       outputs/evaluations/{CONTROL_MODEL_SHORT}/generation_scores.json \
    --model        gpt-4.1-mini \
    --sample_for_manual_audit 5

In [ ]:
# Print control generation results table
scores = json.load(open(f'outputs/evaluations/{CONTROL_MODEL_SHORT}/generation_scores.json'))
print(f"{'coeff':>8}  {'correctness':>12}  {'mech_depth':>12}  {'misconception':>14}  {'clarity':>10}")
print('-' * 65)
for row in scores.get('by_coefficient', []):
    print(f"{row['coefficient']:>8}  {row['scientific_correctness']:>12.2f}  "
          f"{row['mechanistic_depth']:>12.2f}  {row['misconception_presence']:>14.2f}  "
          f"{row['clarity']:>10.2f}")

---
## Phase 12 — SAE Bridge (REQUIRED)
Projects the 1.7B LAT vectors onto the SAE decoder rows from `CRISMechInterp/Qwen3_SAEs`. Produces the alignment spectrum (Figure 3) and top-k feature list.

In [ ]:
# SAE bridge only works with 1.7B vectors (SAEs were trained on 1.7B)
SAE_MODEL_SHORT = 'qwen3_1p7b'
assert os.path.exists(f'outputs/vectors/{SAE_MODEL_SHORT}'), \
    'No 1.7B vectors — run Phases 7+8 for 1.7B first'

In [ ]:
!python -m src.analysis.sae_alignment \
    --lat_vectors  outputs/vectors/{SAE_MODEL_SHORT} \
    --sae_repo     CRISMechInterp/Qwen3_SAEs \
    --sae_subfolders qwen3_1p7b_middle_btop128_exp4 qwen3_1p7b_middle_btop256_exp4 \
    --sae_layer_hint hook_resid_post \
    --top_k        20 \
    --output_dir   outputs/sae_alignment/{SAE_MODEL_SHORT}

In [ ]:
# Display alignment spectra inline
from IPython.display import Image, display
import glob

spectrum_plots = glob.glob(f'outputs/sae_alignment/{SAE_MODEL_SHORT}/**/*_spectrum.png', recursive=True)
for p in sorted(spectrum_plots):
    print(p)
    display(Image(p))

In [ ]:
# Print top-20 aligned features per dataset per SAE
top_feature_files = glob.glob(f'outputs/sae_alignment/{SAE_MODEL_SHORT}/**/*_top_features.json', recursive=True)
for f in sorted(top_feature_files):
    d = json.load(open(f))
    print(f"\n{f}")
    for feat in d[:5]:   # print top 5
        print(f"  rank {feat['rank']:2d}  feature {feat['feature_idx']:5d}  |cos|={feat['cos']:.4f}  sign={feat['sign']:+d}")

---
## Phase 13 — Scaling Plots + Bootstrap CIs
Run this cell **after both 1.7B and 4B evaluations are complete**.

In [ ]:
!python -m src.analysis.plot_scaling_trends \
    --evaluations_dir outputs/evaluations \
    --output_dir      outputs/plots \
    --n_boot          1000

# Display scaling plots inline
plots = glob.glob('outputs/plots/scaling*.png')
for p in sorted(plots):
    print(p)
    display(Image(p))

---
## Push results back to GitHub

In [ ]:
# Configure git (first time only)
!git config user.email 'colab@repe'
!git config user.name  'Colab'

# Stage everything except large .pt files (those stay on Drive)
!echo '*.pt' >> .gitignore
!git add outputs/evaluations/ outputs/plots/ outputs/sae_alignment/ \
          outputs/generations/ data/processed/ .gitignore
!git commit -m 'Colab: Phases 7-13 results'

# Push using token embedded in URL (add HF_TOKEN or a GitHub PAT to Colab secrets)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_AUTH = f'https://{GITHUB_TOKEN}@github.com/TanmayAgarwal123/CRIS_Labs_Spring_2026.git'
!git push {REPO_AUTH} repe-science-emergence